In [ ]:
#!/usr/bin/env python3
"""periodic_grf_sdf.py — Step 1: Periodic GRF → SDF Pipeline

Generates random 2D periodic unit-cell geometries using Gaussian random fields
and computes their signed distance function (SDF) representation.

Reference: Liu et al., "Toward Signed Distance Function Based Metamaterial
Design," CMAME (2025).
GitHub:    https://github.com/QibangLiu/SDFGeoDesign
"""

from __future__ import annotations

import numpy as np
from numpy.typing import NDArray
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator
from skimage.measure import find_contours
import shapely
from shapely.geometry import MultiLineString
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Tunable parameters (defaults) — aligned with Liu et al. reference impl.
# ──────────────────────────────────────────────────────────────────────────────
GRF_RESOLUTION: int = 64       # Grid resolution for GRF generation
SDF_RESOLUTION: int = 120      # Grid resolution for SDF evaluation
SDF_DOMAIN_PAD: float = 0.1    # Padding beyond [0,1]^2 for the SDF domain
LEN_SCALE: float = 0.15        # Correlation length of Gaussian covariance
VARIANCE: float = 10.0         # Variance (sill) of the GRF
GRF_MODE_NO: int = 32          # Number of Fourier modes per dim (gstools)
THRESHOLD_SCALE: float = 0.4   # Fraction of field range for threshold window
VFRAC_MIN: float = 0.15        # Minimum allowed solid volume fraction
VFRAC_MAX: float = 0.85        # Maximum allowed solid volume fraction
PRINT_TOL: float = 0.08        # Manufacturing tolerance (fraction of domain)
MIN_HOLE_RADIUS: float = 0.04  # Min equivalent radius for interior holes
NARROW_THRESHOLD: float = 0.024  # Min dist between non-adjacent contour pts
MAX_RETRIES: int = 50          # Max consecutive rejections before error


# ──────────────────────────────────────────────────────────────────────────────
# 1. GRF Generation
# ──────────────────────────────────────────────────────────────────────────────
def generate_periodic_grf(
    resolution: int = GRF_RESOLUTION,
    len_scale: float = LEN_SCALE,
    variance: float = VARIANCE,
    mode_no: int = GRF_MODE_NO,
    rng: np.random.Generator | None = None,
) -> NDArray[np.float64]:
    """Generate a 2D periodic Gaussian random field on [0, 1)^2.

    Uses gstools with the explicit Fourier generator (``generator="Fourier"``,
    ``mode_no=32``) for periodic fields, matching the reference implementation.
    Falls back to a manual spectral method if gstools is unavailable.

    Parameters
    ----------
    resolution : int
        Number of grid points along each axis.
    len_scale : float
        Correlation length of the Gaussian covariance model.
    variance : float
        Variance (sill) of the covariance model.
    mode_no : int
        Number of Fourier modes per dimension (gstools Fourier generator).
    rng : numpy.random.Generator or None
        Random number generator for reproducibility.

    Returns
    -------
    field : ndarray of shape (resolution, resolution)
        Periodic GRF values on a uniform grid.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.linspace(0, 1, resolution, endpoint=False)

    try:
        import gstools as gs

        model = gs.Gaussian(dim=2, var=variance, len_scale=len_scale)
        srf = gs.SRF(
            model,
            generator="Fourier",
            period=[1.0, 1.0],
            mode_no=mode_no,
            seed=int(rng.integers(0, 2**31)),
        )
        field = srf.structured([x, x])
        return np.asarray(field, dtype=np.float64)
    except Exception:
        return _generate_periodic_grf_manual(resolution, len_scale, variance, rng)


def _generate_periodic_grf_manual(
    resolution: int,
    len_scale: float,
    variance: float,
    rng: np.random.Generator,
) -> NDArray[np.float64]:
    """Fallback: periodic GRF via the Fourier spectral method.

    Uses the representation:
        U(x) = sum_{k != 0} sqrt(S(k) * dk^2) * (Z1 cos(k.x) + Z2 sin(k.x))
    where k = 2*pi*(n1, n2) for integer n1, n2, dk = 2*pi, and S(k) is the
    power spectrum of the Gaussian covariance C(r) = var * exp(-(r/l)^2):
        S(k) = var * pi * l^2 * exp(-|k|^2 * l^2 / 4)
    """
    x = np.linspace(0, 1, resolution, endpoint=False)
    xx, yy = np.meshgrid(x, x, indexing="ij")

    # Mode cutoff: include modes where S(k) > ~1e-8 * S(0)
    k_max = np.sqrt(4.0 * 8.0 * np.log(10.0)) / len_scale
    n_max = int(np.ceil(k_max / (2.0 * np.pi))) + 1
    n_max = max(n_max, 3)

    dk2 = (2.0 * np.pi) ** 2  # area element in k-space

    field = np.zeros((resolution, resolution), dtype=np.float64)

    for n1 in range(-n_max, n_max + 1):
        for n2 in range(-n_max, n_max + 1):
            if n1 == 0 and n2 == 0:
                continue
            kx = 2.0 * np.pi * n1
            ky = 2.0 * np.pi * n2
            k_sq = kx**2 + ky**2

            # Gaussian power spectrum (2D FT of Gaussian covariance)
            S_k = variance * np.pi * len_scale**2 * np.exp(
                -k_sq * len_scale**2 / 4.0
            )
            amplitude = np.sqrt(S_k * dk2)

            z1 = rng.standard_normal()
            z2 = rng.standard_normal()
            phase = kx * xx + ky * yy
            field += amplitude * (z1 * np.cos(phase) + z2 * np.sin(phase))

    return field

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2. Thresholding — adaptive, following reference implementation
# ──────────────────────────────────────────────────────────────────────────────
def threshold_grf(
    grf: NDArray[np.float64],
    threshold: float | None = None,
    rng: np.random.Generator | None = None,
    threshold_scale: float = THRESHOLD_SCALE,
) -> tuple[NDArray[np.bool_], float]:
    """Threshold a GRF to produce a binary solid/void map.

    The threshold is chosen adaptively relative to the field's value range,
    matching the reference implementation::

        ave   = (max + min) / 2
        scale = threshold_scale * (max - min)
        v     ~ Uniform(ave, ave + scale)

    This ensures the threshold always falls within the field's actual values
    and biases toward removing the lower-value region.

    Parameters
    ----------
    grf : ndarray
        The Gaussian random field.
    threshold : float or None
        If None, drawn adaptively from the field range.
    rng : Generator or None
        Random number generator.
    threshold_scale : float
        Fraction of (max - min) defining the threshold window width.

    Returns
    -------
    binary : ndarray of bool
        True where solid (GRF >= threshold), False where void.
    threshold_used : float
        The threshold value that was applied.
    """
    if rng is None:
        rng = np.random.default_rng()
    if threshold is None:
        field_min = float(grf.min())
        field_max = float(grf.max())
        ave = (field_max + field_min) / 2.0
        scale = threshold_scale * (field_max - field_min)
        threshold = float(rng.uniform(ave, ave + scale))
    binary: NDArray[np.bool_] = grf >= threshold
    return binary, threshold

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3. Contour Extraction
# ──────────────────────────────────────────────────────────────────────────────
def extract_contours(
    grf: NDArray[np.float64],
    threshold: float,
    resolution: int = GRF_RESOLUTION,
) -> list[NDArray[np.float64]]:
    """Extract boundary contours from the GRF at the given threshold level.

    Uses ``skimage.measure.find_contours`` with ``positive_orientation='high'``
    (matching the reference — contours are oriented so that the high side of
    the field is to the right) and converts pixel coordinates to physical
    coordinates in [0, 1)^2.

    Parameters
    ----------
    grf : ndarray of shape (resolution, resolution)
        The raw GRF field.
    threshold : float
        Level at which to extract contours.
    resolution : int
        Grid resolution (used for coordinate scaling).

    Returns
    -------
    contours : list of ndarray
        Each array has shape (M, 2) with (x, y) in physical coordinates.
    """
    raw_contours = find_contours(grf, level=threshold, positive_orientation="high")
    # Pixel index i corresponds to physical coordinate i / resolution
    scale = 1.0 / resolution
    contours = [c * scale for c in raw_contours]
    return contours

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Signed Distance Function — Shapely exact distances + GRF interpolation
# ──────────────────────────────────────────────────────────────────────────────
def compute_sdf(
    contours: list[NDArray[np.float64]],
    grf: NDArray[np.float64],
    threshold: float,
    sdf_resolution: int = SDF_RESOLUTION,
    domain_pad: float = SDF_DOMAIN_PAD,
    grf_resolution: int = GRF_RESOLUTION,
) -> NDArray[np.float64]:
    """Compute the signed distance function on an extended grid.

    * **Distance** is computed via Shapely ``MultiLineString.distance``,
      giving exact shortest distance to boundary *line segments* (not just
      sampled contour points), matching the reference implementation.
    * **Sign** is determined by bilinear interpolation of the GRF to the SDF
      grid and comparing against the threshold (solid where GRF >= threshold).
      This is more accurate near the boundary than a nearest-neighbour lookup
      on the 64x64 binary map.

    Parameters
    ----------
    contours : list of ndarray
        Boundary contours in physical coordinates.
    grf : ndarray of shape (grf_resolution, grf_resolution)
        The raw GRF field (used for sign via interpolation).
    threshold : float
        The GRF threshold (solid where GRF >= threshold).
    sdf_resolution : int
        Number of grid points for the SDF grid.
    domain_pad : float
        Padding beyond [0,1]^2 on each side.
    grf_resolution : int
        Resolution of the GRF grid.

    Returns
    -------
    sdf : ndarray of shape (sdf_resolution, sdf_resolution)
        Signed distance values (negative inside solid, positive outside).
    """
    lo = -domain_pad
    hi = 1.0 + domain_pad
    sx = np.linspace(lo, hi, sdf_resolution)
    sy = np.linspace(lo, hi, sdf_resolution)
    sxx, syy = np.meshgrid(sx, sy, indexing="ij")
    grid_points = np.column_stack([sxx.ravel(), syy.ravel()])

    # ── Distance via Shapely ──────────────────────────────────────────────
    # Tile contour line segments periodically (3x3) so distances near the
    # domain boundary are computed correctly.
    offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),  (0, 0),  (0, 1),
        (1, -1),  (1, 0),  (1, 1),
    ]
    all_line_coords: list[list[list[float]]] = []
    for contour in contours:
        if len(contour) < 2:
            continue
        for dx, dy in offsets:
            shifted = contour + np.array([dx, dy], dtype=np.float64)
            all_line_coords.append(shifted.tolist())

    if len(all_line_coords) == 0:
        # No contours — field is entirely solid or void
        is_solid = grf.mean() >= threshold
        return np.full(
            (sdf_resolution, sdf_resolution),
            -1.0 if is_solid else 1.0,
            dtype=np.float64,
        )

    boundary = MultiLineString(all_line_coords)

    # Vectorised distance (shapely >= 2.0)
    try:
        pts_geom = shapely.points(grid_points)
        distances = np.asarray(shapely.distance(pts_geom, boundary))
    except (AttributeError, TypeError):
        # Fallback for older shapely
        from shapely.geometry import Point
        distances = np.array([boundary.distance(Point(p)) for p in grid_points])

    distances = distances.reshape(sdf_resolution, sdf_resolution)

    # ── Sign via bilinear GRF interpolation ───────────────────────────────
    # Pad GRF by one cell (periodic wrap) so interpolation covers [0, 1].
    x_grf = np.linspace(0, 1, grf_resolution, endpoint=False)
    grf_padded = np.pad(grf, ((0, 1), (0, 1)), mode="wrap")
    x_padded = np.append(x_grf, 1.0)

    interp = RegularGridInterpolator(
        (x_padded, x_padded),
        grf_padded,
        method="linear",
        bounds_error=False,
        fill_value=None,
    )
    wrapped = np.column_stack([sxx.ravel() % 1.0, syy.ravel() % 1.0])
    grf_interp = interp(wrapped).reshape(sdf_resolution, sdf_resolution)
    is_solid = grf_interp >= threshold

    # Negative inside solid, positive outside
    sdf = np.where(is_solid, -distances, distances)
    return sdf

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 5. Validation Filters — aligned with reference implementation
# ──────────────────────────────────────────────────────────────────────────────
def check_periodic_connectivity(binary: NDArray[np.bool_]) -> bool:
    """Check that all solid pixels form a single connected component under
    periodic (wrapping) boundary conditions.

    Tiles the binary map 3x3, labels connected components, and verifies that
    every solid pixel in the centre tile has the same label.
    """
    tiled = np.tile(binary, (3, 3))
    labeled, num_features = ndimage.label(tiled)
    if num_features == 0:
        return False
    n = binary.shape[0]
    center = labeled[n : 2 * n, n : 2 * n]
    solid_labels = center[binary]
    if len(solid_labels) == 0:
        return False
    return len(np.unique(solid_labels)) == 1


def _contour_arc_length(contour: NDArray[np.float64]) -> float:
    """Compute the total arc length of a contour."""
    diffs = np.diff(contour, axis=0)
    return float(np.sqrt((diffs**2).sum(axis=1)).sum())


def _contour_is_closed(contour: NDArray[np.float64], tol: float = 1e-3) -> bool:
    """Check whether first and last points of a contour coincide."""
    return bool(np.linalg.norm(contour[0] - contour[-1]) < tol)


def _contour_enclosed_area(contour: NDArray[np.float64]) -> float:
    """Enclosed area via the shoelace formula (absolute value)."""
    x, y = contour[:, 0], contour[:, 1]
    return float(
        0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
    )


def _contour_touches_boundary(
    contour: NDArray[np.float64],
    tol: float | None = None,
    resolution: int = GRF_RESOLUTION,
) -> bool:
    """Check if any point of a contour is within *tol* of the [0, 1) boundary."""
    if tol is None:
        tol = 1.5 / resolution
    return bool(contour.min() < tol or contour.max() > 1.0 - tol)


def _has_narrow_regions(
    contours: list[NDArray[np.float64]],
    threshold: float = NARROW_THRESHOLD,
    min_index_gap: int = 5,
) -> bool:
    """Detect narrow regions: non-adjacent contour points closer than *threshold*.

    For points on the **same** contour they must be separated by at least
    *min_index_gap* indices (accounting for periodic wrapping of the contour)
    to be considered non-adjacent.  For points on **different** contours any
    close pair triggers rejection.  Mirrors the reference ``has_narrow_regions``.
    """
    from scipy.spatial import cKDTree

    for i, c1 in enumerate(contours):
        if len(c1) < 2 * min_index_gap:
            continue
        for j in range(i, len(contours)):
            c2 = contours[j]
            if len(c2) < 2:
                continue
            if i == j:
                # Same contour — check non-adjacent pairs
                tree = cKDTree(c1)
                for k in range(len(c1)):
                    idxs = tree.query_ball_point(c1[k], threshold)
                    for m in idxs:
                        gap = abs(k - m)
                        gap = min(gap, len(c1) - gap)  # periodic gap
                        if gap > min_index_gap:
                            return True
            else:
                # Different contours — any close pair
                tree = cKDTree(c2)
                dists, _ = tree.query(c1)
                if dists.min() < threshold:
                    return True
    return False


def validate_sample(
    binary: NDArray[np.bool_],
    contours: list[NDArray[np.float64]],
    resolution: int = GRF_RESOLUTION,
    vfrac_min: float = VFRAC_MIN,
    vfrac_max: float = VFRAC_MAX,
    print_tol: float = PRINT_TOL,
    min_hole_radius: float = MIN_HOLE_RADIUS,
    narrow_threshold: float = NARROW_THRESHOLD,
) -> bool:
    """Apply validity filters to a generated sample.

    Checks (aligned with the reference where applicable):
    1. Volume fraction within [vfrac_min, vfrac_max].
    2. Single connected solid region (periodic connectivity).
    3. Closed contours too close to domain boundary -> reject.
    4. Interior holes with equivalent radius < min_hole_radius -> reject.
    5. Narrow regions detected between non-adjacent contour points -> reject.

    Parameters
    ----------
    binary : ndarray of bool
        Binary solid/void map.
    contours : list of ndarray
        Extracted boundary contours in physical coordinates.
    resolution : int
        GRF grid resolution.
    vfrac_min, vfrac_max : float
        Allowed volume fraction bounds.
    print_tol : float
        Manufacturing tolerance (domain fraction).
    min_hole_radius : float
        Minimum equivalent radius for interior holes.
    narrow_threshold : float
        Minimum allowed distance between non-adjacent contour points.

    Returns
    -------
    valid : bool
    """
    # 1. Volume fraction
    vfrac = binary.mean()
    if vfrac < vfrac_min or vfrac > vfrac_max:
        return False

    # 2. Periodic connectivity
    if not check_periodic_connectivity(binary):
        return False

    if len(contours) == 0:
        return False

    # 3 & 4. Per-contour checks
    boundary_tol = 0.5 * print_tol  # reference uses fac = 0.5 for closed contours
    for c in contours:
        if _contour_is_closed(c):
            # Reject if a closed contour sits too close to the domain boundary
            if _contour_touches_boundary(c, tol=boundary_tol):
                return False
            # Reject if the enclosed feature is too small
            area = _contour_enclosed_area(c)
            equiv_r = np.sqrt(area / np.pi)
            if equiv_r < min_hole_radius:
                return False

    # 5. Narrow region detection
    if _has_narrow_regions(contours, threshold=narrow_threshold):
        return False

    return True

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 6. Single sample + batch generation
# ──────────────────────────────────────────────────────────────────────────────
def generate_single_sample(
    rng: np.random.Generator,
    resolution: int = GRF_RESOLUTION,
    sdf_resolution: int = SDF_RESOLUTION,
    len_scale: float = LEN_SCALE,
    variance: float = VARIANCE,
    mode_no: int = GRF_MODE_NO,
    threshold_scale: float = THRESHOLD_SCALE,
    domain_pad: float = SDF_DOMAIN_PAD,
) -> dict | None:
    """Generate one sample.  Returns None if validation fails.

    Returns
    -------
    sample : dict or None
        Keys: grf, binary, contours, sdf, volume_fraction, threshold.
    """
    grf = generate_periodic_grf(resolution, len_scale, variance, mode_no, rng)
    binary, thresh = threshold_grf(grf, rng=rng, threshold_scale=threshold_scale)
    contours = extract_contours(grf, thresh, resolution)

    if not validate_sample(binary, contours, resolution):
        return None

    sdf = compute_sdf(contours, grf, thresh, sdf_resolution, domain_pad, resolution)

    return {
        "grf": grf,
        "binary": binary,
        "contours": contours,
        "sdf": sdf,
        "volume_fraction": float(binary.mean()),
        "threshold": thresh,
    }


def generate_dataset(
    n_samples: int,
    seed: int = 42,
    resolution: int = GRF_RESOLUTION,
    sdf_resolution: int = SDF_RESOLUTION,
    len_scale: float = LEN_SCALE,
    variance: float = VARIANCE,
    mode_no: int = GRF_MODE_NO,
    threshold_scale: float = THRESHOLD_SCALE,
    domain_pad: float = SDF_DOMAIN_PAD,
    max_retries: int = MAX_RETRIES,
) -> dict:
    """Generate a dataset of valid periodic unit-cell SDF samples.

    Parameters
    ----------
    n_samples : int
        Number of valid samples to produce.
    seed : int
        Random seed for reproducibility.
    resolution : int
        GRF grid resolution.
    sdf_resolution : int
        SDF grid resolution.
    len_scale : float
        GRF correlation length.
    variance : float
        GRF variance.
    mode_no : int
        Number of Fourier modes for gstools.
    threshold_scale : float
        Fraction of field range for adaptive threshold window.
    domain_pad : float
        SDF domain padding beyond [0,1]^2.
    max_retries : int
        Maximum consecutive rejections before raising RuntimeError.

    Returns
    -------
    dataset : dict
        "sdf"             : ndarray (n_samples, sdf_resolution, sdf_resolution)
        "binary"          : ndarray (n_samples, resolution, resolution)
        "contours"        : list[list[ndarray]]
        "volume_fraction" : ndarray (n_samples,)
        "threshold"       : ndarray (n_samples,)
        "grf"             : ndarray (n_samples, resolution, resolution)
        "n_rejected"      : int — total number of rejected samples
    """
    rng = np.random.default_rng(seed)

    sdfs: list[NDArray] = []
    binaries: list[NDArray] = []
    all_contours: list[list[NDArray]] = []
    vfracs: list[float] = []
    thresholds: list[float] = []
    grfs: list[NDArray] = []
    n_rejected = 0
    consecutive_rejects = 0

    while len(sdfs) < n_samples:
        sample = generate_single_sample(
            rng, resolution, sdf_resolution, len_scale, variance,
            mode_no, threshold_scale, domain_pad,
        )
        if sample is None:
            n_rejected += 1
            consecutive_rejects += 1
            if consecutive_rejects >= max_retries:
                raise RuntimeError(
                    f"Failed after {max_retries} consecutive rejections. "
                    f"Generated {len(sdfs)}/{n_samples} samples so far. "
                    f"Consider adjusting len_scale, threshold_scale, or filters."
                )
            continue

        consecutive_rejects = 0
        sdfs.append(sample["sdf"])
        binaries.append(sample["binary"])
        all_contours.append(sample["contours"])
        vfracs.append(sample["volume_fraction"])
        thresholds.append(sample["threshold"])
        grfs.append(sample["grf"])

        print(
            f"  [{len(sdfs):3d}/{n_samples}] "
            f"VF={sample['volume_fraction']:.3f}  "
            f"thr={sample['threshold']:+.3f}  "
            f"(rejected so far: {n_rejected})"
        )

    return {
        "sdf": np.stack(sdfs),
        "binary": np.stack(binaries),
        "contours": all_contours,
        "volume_fraction": np.array(vfracs),
        "threshold": np.array(thresholds),
        "grf": np.stack(grfs),
        "n_rejected": n_rejected,
    }

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
# 7. Visualization
# ──────────────────────────────────────────────────────────────────────────────
def plot_sdf_gallery(
    dataset: dict,
    sdf_resolution: int = SDF_RESOLUTION,
    domain_pad: float = SDF_DOMAIN_PAD,
    filename: str = "sdf_gallery.png",
) -> None:
    """4x4 gallery of SDF fields with zero contours overlaid in red."""
    n = min(16, dataset["sdf"].shape[0])
    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    lo, hi = -domain_pad, 1.0 + domain_pad

    for idx in range(16):
        ax = axes[idx // 4, idx % 4]
        if idx < n:
            sdf = dataset["sdf"][idx]
            vf = dataset["volume_fraction"][idx]
            vmax = np.abs(sdf).max()
            ax.imshow(
                sdf.T, origin="lower", extent=[lo, hi, lo, hi],
                cmap="RdBu", vmin=-vmax, vmax=vmax,
            )
            ax.contour(
                np.linspace(lo, hi, sdf.shape[0]),
                np.linspace(lo, hi, sdf.shape[1]),
                sdf.T, levels=[0.0], colors="red", linewidths=1.2,
            )
            ax.set_title(f"VF = {vf:.3f}", fontsize=9)
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.set_aspect("equal")
        else:
            ax.axis("off")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("SDF Gallery (zero contour in red)", fontsize=14)
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    print(f"Saved {filename}")


def plot_grf_gallery(dataset: dict, filename: str = "grf_gallery.png") -> None:
    """4x4 gallery of raw GRF fields with threshold contours overlaid."""
    n = min(16, dataset["grf"].shape[0])
    fig, axes = plt.subplots(4, 4, figsize=(14, 14))

    for idx in range(16):
        ax = axes[idx // 4, idx % 4]
        if idx < n:
            grf = dataset["grf"][idx]
            thresh = dataset["threshold"][idx]
            res = grf.shape[0]
            ax.imshow(grf.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
            ax.contour(
                np.linspace(0, 1, res),
                np.linspace(0, 1, res),
                grf.T, levels=[thresh], colors="red", linewidths=1.2,
            )
            ax.set_title(f"thr = {thresh:+.3f}", fontsize=9)
            ax.set_aspect("equal")
        else:
            ax.axis("off")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("GRF Gallery (threshold contour in red)", fontsize=14)
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    print(f"Saved {filename}")


def plot_sample_detail(
    dataset: dict,
    sample_idx: int = 0,
    sdf_resolution: int = SDF_RESOLUTION,
    domain_pad: float = SDF_DOMAIN_PAD,
    filename: str = "sample_detail.png",
) -> None:
    """Detailed 4-panel figure for a single sample."""
    grf = dataset["grf"][sample_idx]
    binary = dataset["binary"][sample_idx]
    sdf = dataset["sdf"][sample_idx]
    thresh = dataset["threshold"][sample_idx]
    vf = dataset["volume_fraction"][sample_idx]
    res = grf.shape[0]

    lo, hi = -domain_pad, 1.0 + domain_pad
    vmax = np.abs(sdf).max()

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # (a) Raw GRF with threshold contour
    ax = axes[0]
    ax.imshow(grf.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
    ax.contour(
        np.linspace(0, 1, res), np.linspace(0, 1, res),
        grf.T, levels=[thresh], colors="red", linewidths=1.5,
    )
    ax.set_title(f"(a) GRF  (thr = {thresh:+.3f})", fontsize=10)
    ax.set_aspect("equal")

    # (b) Binary solid/void
    ax = axes[1]
    ax.imshow(binary.T.astype(float), origin="lower", extent=[0, 1, 0, 1],
              cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"(b) Binary  (VF = {vf:.3f})", fontsize=10)
    ax.set_aspect("equal")

    # (c) SDF with zero contour
    ax = axes[2]
    ax.imshow(
        sdf.T, origin="lower", extent=[lo, hi, lo, hi],
        cmap="RdBu", vmin=-vmax, vmax=vmax,
    )
    ax.contour(
        np.linspace(lo, hi, sdf.shape[0]),
        np.linspace(lo, hi, sdf.shape[1]),
        sdf.T, levels=[0.0], colors="red", linewidths=1.5,
    )
    ax.set_title("(c) SDF  (zero contour)", fontsize=10)
    ax.set_aspect("equal")

    # (d) Tiled 3x3 SDF for periodicity verification
    ax = axes[3]
    sx = np.linspace(lo, hi, sdf.shape[0])
    sy = np.linspace(lo, hi, sdf.shape[1])
    mask_x = (sx >= 0.0) & (sx <= 1.0)
    mask_y = (sy >= 0.0) & (sy <= 1.0)
    sdf_unit = sdf[np.ix_(mask_x, mask_y)]
    sdf_tiled = np.tile(sdf_unit, (3, 3))
    ax.imshow(
        sdf_tiled.T, origin="lower", extent=[0, 3, 0, 3],
        cmap="RdBu", vmin=-vmax, vmax=vmax,
    )
    for i in range(1, 3):
        ax.axhline(i, color="k", linewidth=0.5, linestyle="--")
        ax.axvline(i, color="k", linewidth=0.5, linestyle="--")
    ax.set_title("(d) SDF tiled 3×3  (periodicity)", fontsize=10)
    ax.set_aspect("equal")

    fig.suptitle(f"Sample {sample_idx} — Detailed View", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {filename}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("Periodic GRF -> SDF Pipeline  (Step 1)")
print("=" * 60)
print(f"GRF resolution : {GRF_RESOLUTION}x{GRF_RESOLUTION}")
print(f"SDF resolution : {SDF_RESOLUTION}x{SDF_RESOLUTION}")
print(f"Correlation len: {LEN_SCALE}")
print(f"Variance       : {VARIANCE}")
print(f"Fourier modes  : {GRF_MODE_NO}")
print(f"Threshold scale: {THRESHOLD_SCALE}")
print()

dataset = generate_dataset(n_samples=16, seed=42)

n_valid = dataset["sdf"].shape[0]
n_rejected = dataset["n_rejected"]
vf = dataset["volume_fraction"]

print()
print("-" * 60)
print("Summary")
print("-" * 60)
print(f"  Valid samples : {n_valid}")
print(f"  Rejected      : {n_rejected}")
print(f"  Volume fraction -- mean: {vf.mean():.3f}, std: {vf.std():.3f}")
print(f"                    min:  {vf.min():.3f}, max: {vf.max():.3f}")
print()

plot_sdf_gallery(dataset, filename="sdf_gallery.png")
plot_grf_gallery(dataset, filename="grf_gallery.png")
plot_sample_detail(dataset, sample_idx=0, filename="sample_detail.png")

print("\nDone.")